# Bài 05 · Gradient descent: đi xuống dốc từng bước, từ một đường cong đến MNIST

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/05-gradient-descent.ipynb)

Notebook đi kèm [Bài 05 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/gradient-descent/) (reel và demo tương tác nằm ở trang bài học).
Bài 04 cho ta một con số đo mô hình sai bao nhiêu: hàm mất mát $\mathcal{L}(\theta)$. Bài này trả lời câu hỏi kế tiếp:
**sửa tham số $\theta$ theo hướng nào, và bước dài bao nhiêu, để con số đó nhỏ đi?** Ta tự viết gradient descent bằng
numpy, trước trên những hàm mất mát vẽ được bằng tay, rồi trên ảnh MNIST thật.

**Bạn sẽ làm:**
1. Đo gradient bằng sai phân hữu hạn và dùng nó để kiểm tra công thức đạo hàm viết tay.
2. Chạy $\theta \leftarrow \theta - \eta\,\nabla\mathcal{L}(\theta)$ trên hàm mất mát của demo với bốn tốc độ học, và tìm
   ngưỡng $\eta < 2/\lambda$.
3. Thấy vì sao gradient không chỉ thẳng vào đáy, và vì sao gradient descent có thể đứng yên ở một điểm yên ngựa.
4. Huấn luyện hồi quy softmax trên MNIST: gradient descent trên cả tập so với mini-batch SGD, bước so với lượt (epoch).
5. Đo nhiễu của gradient lô theo cỡ lô $B$, rồi huấn luyện mạng 784 → 128 → 64 → 10 của khóa học bằng cùng vòng lặp.

**Cần biết trước:** [Bài 04 · Softmax & hàm mất mát](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-mat-mat/)
(softmax, cross-entropy) và quy ước $z = xW + b$ của [Bài 01 · Perceptron](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/perceptron/).
Đạo hàm ở mức "độ dốc của tiếp tuyến" là đủ. **Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải MNIST (~11 MB) lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra, và trùng với số trong demo và dữ liệu huấn luyện của khóa học.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân, dấu chấm hàng
nghìn; `sci()` in số rất nhỏ dạng 3,2·10⁻¹¹. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import gzip
import hashlib
import itertools
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38."""
    s = f"{x:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """Số rất nhỏ hoặc rất lớn: 3,2·10⁻¹¹."""
    m, e = f"{x:.{digits}e}".split("e")
    return f"{vn(float(m), digits)}·10{str(int(e)).translate(SUP)}"


print(vn(60000, 0), vn(0.15), vn(-2.0879, 3), pct(0.9089), sci(3.2e-11))

## 2 · Dữ liệu MNIST

Mục 3–6 chạy trên những hàm mất mát nhỏ có công thức; từ mục 7 ta dùng MNIST (LeCun và cộng sự, 1998): ảnh chữ số viết
tay xám 28×28. Ô dưới tải 4 tệp gốc và kiểm tra mã MD5 như Bài 01, rồi trải mỗi ảnh thành một hàng $x$ gồm 784 số trong
$[0, 1]$. Cả tập huấn luyện là ma trận $X$: mỗi hàng một ảnh.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
Xtr, ytr = train_img.reshape(-1, 784) / 255.0, train_lab.astype(int)  # mỗi hàng là một ảnh x (vectơ hàng 1×784)
Xte, yte = test_img.reshape(-1, 784) / 255.0, test_lab.astype(int)
N = len(Xtr)
print(f"huấn luyện: {vn(N, 0)} ảnh · kiểm tra: {vn(len(Xte), 0)} ảnh · tổng {vn(N + len(Xte), 0)}")
print("X huấn luyện có cỡ", Xtr.shape)
assert (N, len(Xte)) == (60000, 10000)

Mười ảnh huấn luyện đầu tiên, mỗi ảnh ghi nhãn đúng ở trên:

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for ax, img, lab in zip(axes, train_img, train_lab):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(lab), fontsize=11)
    ax.axis("off")
plt.show()

## 3 · Độ dốc tại một điểm chính là gradient

Hãy tưởng tượng một người leo núi trong sương mù: không thấy đáy thung lũng ở đâu, chỉ cảm được mặt đất dưới chân dốc
lên về phía nào. Gradient descent cũng vậy: nó chỉ biết **độ dốc tại điểm đang đứng**.

Demo của bài dùng một hàm mất mát một chiều có hai thung lũng:

$$\mathcal{L}(\theta) = 0{,}1\,\theta^4 - 0{,}8\,\theta^2 + 0{,}3\,\theta + 2{,}5, \qquad
\mathcal{L}'(\theta) = 0{,}4\,\theta^3 - 1{,}6\,\theta + 0{,}3.$$

Đạo hàm $\mathcal{L}'(\theta)$ là độ dốc của tiếp tuyến. Không cần công thức ta vẫn đo được nó: nhích $\theta$ một đoạn
nhỏ $h$ sang hai phía và xem $\mathcal{L}$ đổi bao nhiêu (**sai phân trung tâm**):

$$\mathcal{L}'(\theta) \approx \frac{\mathcal{L}(\theta + h) - \mathcal{L}(\theta - h)}{2h}.$$

Khi có nhiều tham số $\theta = (\theta_1, \dots, \theta_n)$, nhích lần lượt từng tọa độ cho $n$ **đạo hàm riêng**; xếp
chúng thành một vectơ ta được **gradient** $\nabla\mathcal{L}(\theta)$. Hàm `num_grad` dưới đây làm việc đó cho mảng bất
kỳ; cả bài dùng nó để kiểm tra mọi công thức gradient viết tay.

In [ ]:
def L(t):
    return 0.1 * t**4 - 0.8 * t**2 + 0.3 * t + 2.5  # hàm mất mát của demo


def dL(t):
    return 0.4 * t**3 - 1.6 * t + 0.3  # đạo hàm viết tay


def num_grad(f, theta, h=1e-5):
    """Gradient bằng sai phân trung tâm: nhích lần lượt từng tọa độ của theta sang hai phía."""
    theta = np.array(theta, dtype=float)
    g = np.zeros_like(theta)
    for i in range(theta.size):
        e = np.zeros_like(theta)
        e.flat[i] = h
        g.flat[i] = (f(theta + e) - f(theta - e)) / (2 * h)
    return g


def L_vec(t):
    return L(t[0])


t0 = 2.6  # điểm xuất phát mặc định của demo
g_num = num_grad(L_vec, [t0])[0]
print(
    f"θ = {vn(t0, 1)}: công thức L′ = {vn(dL(t0), 4)}, sai phân trung tâm = {vn(g_num, 4)} (lệch {sci(abs(g_num - dL(t0)))})"
)
assert abs(g_num - dL(t0)) < 1e-8

Sai phân khớp công thức tới 10 chữ số. Vì sao `num_grad` chọn $h = 10^{-5}$? Ô dưới thử nhiều $h$:

In [ ]:
for h in [1e-1, 1e-3, 1e-5, 1e-7, 1e-9, 1e-11]:
    err = abs((L(t0 + h) - L(t0 - h)) / (2 * h) - dL(t0))
    print(f"h = {sci(h, 0):>6}:  sai số {sci(err)}")

Sai số giảm như $h^2$ khi $h$ nhỏ dần (mỗi lần chia $h$ cho 100, sai số giảm khoảng 10.000 lần), rồi lại **tăng** khi $h$
quá nhỏ: số thực của máy tính chỉ giữ khoảng 16 chữ số, nên $\mathcal{L}(\theta + h) - \mathcal{L}(\theta - h)$ mất gần hết
chữ số có nghĩa. $h$ không phải càng nhỏ càng tốt; $10^{-5}$ nằm ở vùng giữa.

Đáy và đỉnh là nơi độ dốc bằng 0. Giải $\mathcal{L}'(\theta) = 0$ (một đa thức bậc 3) cho ba điểm; dấu của đạo hàm bậc
hai $\mathcal{L}''(\theta) = 1{,}2\,\theta^2 - 1{,}6$ (độ cong) cho biết đó là đáy hay đỉnh.

In [ ]:
roots = np.sort(np.roots([0.4, 0, -1.6, 0.3]).real)  # ba nghiệm của L′(θ) = 0
d2 = 1.2 * roots**2 - 1.6  # L″(θ): dương là đáy, âm là đỉnh
for r, c in zip(roots, d2):
    print(f"θ = {vn(r, 3):>6}:  L = {vn(L(r), 3)},  L″ = {vn(c, 3):>6}  →  {'cực tiểu' if c > 0 else 'cực đại'}")
assert np.round(roots[[0, 2]], 3).tolist() == [-2.088, 1.899], "khác với demo: hãy kiểm tra lại hàm L"

Đúng hai đáy mà demo đánh dấu: **cực tiểu toàn cục** $\theta \approx -2{,}088$ (mất mát thấp nhất) và **cực tiểu cục bộ**
$\theta \approx 1{,}899$ (demo gọi là cực tiểu địa phương), ngăn cách bởi một đỉnh ở $\theta \approx 0{,}189$.

Hình dưới vẽ $\mathcal{L}(\theta)$ (đường xanh), tiếp tuyến tại bốn điểm (gạch xám) và mũi tên cam chỉ hướng
$-\mathcal{L}'$, tức hướng xuống dốc. Đáy toàn cục là chấm tròn, đáy cục bộ là ô vuông, đỉnh là tam giác.

In [ ]:
grid = np.linspace(-3, 3, 400)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(grid, L(grid), color=BLUE, lw=2.5, label="L(θ)")
for p in [-2.6, -0.8, 1.0, 2.6]:
    s = dL(p)
    tt = np.array([p - 0.45, p + 0.45])
    ax.plot(tt, L(p) + s * (tt - p), color=GREY, ls="--", lw=1.5)
    ax.annotate(
        "", xy=(p - 0.5 * np.sign(s), L(p)), xytext=(p, L(p)), arrowprops={"arrowstyle": "->", "color": ORANGE, "lw": 2}
    )
    ax.text(p, L(p) + 0.35, f"L′ = {vn(s, 2)}", ha="center", fontsize=9)
for r, m, name in zip(roots, ["o", "^", "s"], ["đáy toàn cục", "đỉnh", "đáy cục bộ"]):
    ax.plot(r, L(r), m, color="black", ms=8)
    ax.text(r, L(r) - 0.45, name, ha="center", fontsize=9)
ax.set(xlabel="θ", ylabel="L(θ)", xlim=(-3, 3), ylim=(-0.3, 5), title="Độ dốc tại một điểm: mũi tên cam là hướng −L′")
plt.show()
print("độ dốc tại bốn điểm:", " · ".join(f"L′({vn(p, 1)}) = {vn(dL(p), 2)}" for p in [-2.6, -0.8, 1.0, 2.6]))

Độ dốc dương (ở $\theta = 2{,}6$) nghĩa là đi sang phải thì lên dốc, nên ta bước sang trái; độ dốc âm thì ngược lại.
Trong mọi trường hợp, bước **ngược dấu** đạo hàm là xuống dốc. Với nhiều chiều cũng vậy: $\nabla\mathcal{L}$ chỉ hướng
mất mát tăng nhanh nhất, nên ta đi theo $-\nabla\mathcal{L}$.

## 4 · Một bước xuống dốc: $\theta \leftarrow \theta - \eta\,\mathcal{L}'(\theta)$

Gradient descent (Cauchy, 1847) lặp lại đúng một việc: đứng ở $\theta$, đo độ dốc, rồi bước ngược chiều nó một đoạn tỉ lệ
với **tốc độ học** $\eta$. Độ dài bước là $\eta\,|\mathcal{L}'(\theta)|$: dốc càng đứng thì bước càng dài, gần đáy dốc thoải
nên bước tự ngắn lại. Hàm `gd_1d` dừng theo đúng luật của demo: **hội tụ** khi $|\mathcal{L}'| < 10^{-3}$ hoặc $\theta$ gần
như không đổi; **phân kỳ** khi $\theta$ văng ra ngoài vùng $|\theta| \le 3{,}4$; và dừng sau 80 bước.

In [ ]:
ESCAPE, MAX_STEPS = 3.4, 80  # như demo


def gd_1d(eta, theta0, max_steps=MAX_STEPS):
    """Gradient descent một chiều. Trả về dãy θ₀, θ₁, … và trạng thái lúc dừng."""
    trail = [theta0]
    while True:
        t = trail[-1]
        if abs(t) > ESCAPE:
            return np.array(trail), "phân kỳ"
        if len(trail) > 1 and (abs(dL(t)) < 1e-3 or abs(t - trail[-2]) < 1e-5):
            return np.array(trail), "hội tụ"
        if len(trail) > max_steps:
            return np.array(trail), "chưa dừng"
        trail.append(t - eta * dL(t))  # một bước gradient descent


t1 = t0 - 0.15 * dL(t0)
print(f"bước đầu với η = 0,15:  θ ← {vn(t0, 3)} − 0,15 × {vn(dL(t0), 3)} = {vn(t1, 3)}")

### 🤔 Dự đoán trước
Xuất phát ở $\theta_0 = 2{,}6$ với $\eta = 0{,}15$ (nút "Vừa" của demo). Gradient descent dừng ở đáy nào: đáy toàn cục
$\theta \approx -2{,}088$ hay đáy cục bộ $\theta \approx 1{,}899$?

In [ ]:
trail, status = gd_1d(0.15, 2.6)
print(f"từ θ₀ = 2,6:  {status} sau {len(trail) - 1} bước, θ = {vn(trail[-1], 3)}, L = {vn(L(trail[-1]), 3)}")
trail_neg, status_neg = gd_1d(0.15, -2.6)
print(
    f"từ θ₀ = −2,6: {status_neg} sau {len(trail_neg) - 1} bước, θ = {vn(trail_neg[-1], 3)}, L = {vn(L(trail_neg[-1]), 3)}"
)
assert (status, len(trail) - 1, round(trail[-1], 2)) == ("hội tụ", 13, 1.9), "khác với demo"

Nó dừng ở **đáy cục bộ**, sau 13 bước, đúng như demo báo. Gradient chỉ là thông tin tại chỗ: từ $\theta_0 = 2{,}6$ mọi độ
dốc trên đường đi đều chỉ về đáy gần nhất, và không có gì cho biết bên kia đỉnh $0{,}189$ còn một thung lũng sâu hơn.
Xuất phát ở $-2{,}6$ thì nó tới đáy toàn cục. Gradient descent tìm *một* điểm có gradient bằng 0, không phải điểm thấp
nhất; điểm xuất phát quyết định nó rơi vào thung lũng nào.

### Ba tốc độ học: rùa bò, vừa đẹp và nổ tung
Demo có bốn nút: Nhỏ 0,02 · Vừa 0,15 · Lớn 0,9 · Quá lớn 1,3. Chạy cả bốn từ cùng $\theta_0 = 2{,}6$:

In [ ]:
PRESETS = {0.02: "Nhỏ", 0.15: "Vừa", 0.9: "Lớn", 1.3: "Quá lớn"}  # bốn nút của demo
runs = {eta: gd_1d(eta, 2.6) for eta in PRESETS}
for eta, (tr, st) in runs.items():
    print(f"η = {vn(eta, 2):>4} ({PRESETS[eta]:>7}): {st:>11} sau {len(tr) - 1:>2} bước, θ cuối = {vn(tr[-1], 3)}")
summary = [(st, len(tr) - 1) for tr, st in runs.values()]
assert summary == [("chưa dừng", 80), ("hội tụ", 13), ("chưa dừng", 80), ("phân kỳ", 3)], "khác với demo"

Mỗi ô của hình dưới là một tốc độ học: đường xanh là $\mathcal{L}(\theta)$, các chấm cam nối nhau là $\theta_0, \theta_1,
\dots$ (chấm to là điểm cuối). Điểm văng ra ngoài khung không được vẽ.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)
for ax, (eta, (tr, st)) in zip(axes.flat, runs.items()):
    ax.plot(grid, L(grid), color=BLUE, lw=2)
    inside = np.abs(tr) <= 3
    ax.plot(tr[inside], L(tr[inside]), "-o", color=ORANGE, ms=4, lw=1, alpha=0.7)
    if abs(tr[-1]) <= 3:
        ax.plot(tr[-1], L(tr[-1]), "o", color=ORANGE, ms=10, mec="white")
    ax.set_title(f"η = {vn(eta, 2)} ({PRESETS[eta]}): {st} sau {len(tr) - 1} bước", fontsize=10)
    ax.set(xlim=(-3, 3), ylim=(0, 5))
for ax in axes[1]:
    ax.set_xlabel("θ")
for ax in axes[:, 0]:
    ax.set_ylabel("L(θ)")
plt.tight_layout()
plt.show()

- **Nhỏ (0,02):** đi đúng hướng nhưng sau 80 bước vẫn chưa tới đáy (cần 126 bước, ô dưới). Đây là "rùa bò".
- **Vừa (0,15):** tới đáy trong 13 bước.
- **Lớn (0,9):** không phân kỳ, nhưng cũng không bao giờ dừng: $\theta$ nhảy loạn qua cả hai thung lũng, rồi rơi vào
  một vòng lặp giữa hai điểm ở hai bên đáy cục bộ.
- **Quá lớn (1,3):** chỉ 3 bước là văng ra ngoài: $2{,}6 \to -1{,}52 \to -3{,}24 \to 7{,}38$. Đây là "nổ tung".

Mục sau giải thích vì sao ranh giới giữa "vừa" và "nhảy qua lại" nằm đúng ở đâu.

In [ ]:
tr_slow, st_slow = gd_1d(0.02, 2.6, max_steps=10_000)
print(f"η = 0,02 cần {len(tr_slow) - 1} bước để {st_slow}")
tr_big, _ = gd_1d(0.9, 2.6, max_steps=4000)
tail = tr_big[-4:]
print("η = 0,9 sau 4.000 bước, bốn θ cuối:", " · ".join(vn(t, 4) for t in tail))
assert np.allclose(tail[:2], tail[2:], atol=1e-9)  # θ lặp lại sau mỗi 2 bước: một vòng 2 điểm
print("η = 1,3:", " → ".join(vn(t, 2) for t in runs[1.3][0]))

Với $\eta = 0{,}9$, sau 4.000 bước $\theta$ vẫn đổi chỗ giữa đúng hai điểm 1,3879 và 2,1540, mãi mãi. Gradient descent
không bảo đảm đi xuống: mỗi bước nhảy qua đáy sang sườn bên kia.

## 5 · Vì sao $\eta < 2/\lambda$: gần đáy, thung lũng nào cũng là một cái bát

Gần một đáy $\theta^*$, đường cong trông như parabol: $\mathcal{L}'(\theta) \approx \lambda\,(\theta - \theta^*)$, với
$\lambda = \mathcal{L}''(\theta^*)$ là độ cong của đáy. Thay vào một bước:

$$\theta_{k+1} - \theta^* \approx (1 - \eta\lambda)\,(\theta_k - \theta^*).$$

Mỗi bước nhân khoảng cách tới đáy với $1 - \eta\lambda$. Nó co lại khi $|1 - \eta\lambda| < 1$, tức
$0 < \eta < 2/\lambda$; nhanh nhất khi $\eta = 1/\lambda$ (hệ số bằng 0). Khi $1 < \eta\lambda < 2$, hệ số âm: $\theta$ vẫn
tới đáy nhưng nhảy qua nhảy lại. Khi $\eta\lambda > 2$, mỗi bước đẩy $\theta$ ra xa hơn. Ô dưới tính ngưỡng $2/\lambda$ cho
hai đáy của demo:

In [ ]:
lam_glob, lam_loc = d2[0], d2[2]  # độ cong ở đáy toàn cục và đáy cục bộ
for name, lam in [("đáy toàn cục", lam_glob), ("đáy cục bộ", lam_loc)]:
    print(f"{name}: λ = L″ = {vn(lam, 3)},  η tốt nhất 1/λ = {vn(1 / lam, 3)},  ngưỡng 2/λ = {vn(2 / lam, 3)}")
for eta in PRESETS:
    print(
        f"η = {vn(eta, 2):>4}:  1 − ηλ ở đáy cục bộ = {vn(1 - eta * lam_loc, 3):>6},  ở đáy toàn cục = {vn(1 - eta * lam_glob, 3):>6}"
    )

Ngưỡng là 0,551 ở đáy toàn cục (đáy hẹp hơn, cong hơn) và 0,734 ở đáy cục bộ. Nút "Lớn" 0,9 vượt cả hai ngưỡng: không đáy
nào giữ được $\theta$, nên nó rơi vào vòng nhảy qua lại vừa thấy. Nút "Vừa" 0,15 cho hệ số 0,591: khoảng cách tới đáy
giảm còn 59% mỗi bước. Nút "Nhỏ" 0,02 cho 0,945: mỗi bước chỉ bớt $1 - 0{,}945 = 5{,}5\%$.

Lý thuyết này chỉ là xấp xỉ gần đáy, nên ta kiểm tra trên chính hàm của demo. Trước hết một cái bẫy: luật dừng của demo
($|\mathcal{L}'| < 10^{-3}$) có thể bị lừa.

In [ ]:
tr_fake, st_fake = gd_1d(0.79, 2.6, max_steps=3000)
t_after = tr_fake[-1]
for _ in range(50):
    t_after -= 0.79 * dL(t_after)
print(
    f"η = 0,79: gd_1d báo “{st_fake}” sau {len(tr_fake) - 1} bước, ở θ = {vn(tr_fake[-1], 3)} (|L′| = {sci(abs(dL(tr_fake[-1])))})"
)
print(f"   nhưng chạy thêm 50 bước thì θ = {vn(t_after, 3)}: nó chỉ tình cờ đi ngang qua gần đáy")
assert st_fake == "hội tụ" and abs(t_after - tr_fake[-1]) > 0.1

Gradient nhỏ ở *một* bước không có nghĩa là đã hội tụ: $\theta$ đang nhảy loạn và tình cờ rơi sát đáy, rồi bước sau lại
văng đi. Vì vậy phép quét dưới đây dùng luật chặt hơn: chạy 20.000 bước cho mọi $\eta$ từ 0,01 đến 1 cùng lúc (numpy
tính cả mảng $\eta$ trong một phép toán), và chỉ tính là hội tụ khi $\theta$ cuối cùng đứng yên ở một đáy
($|\mathcal{L}'| < 10^{-6}$).

In [ ]:
etas = np.round(np.arange(0.01, 1.0001, 0.01), 2)
for t_start, lam in [(2.6, lam_loc), (-2.6, lam_glob)]:
    th = np.full(etas.shape, t_start)
    for _ in range(20_000):
        th = th - etas * dL(th)  # 100 lần gradient descent song song, mỗi phần tử một η
        th[np.abs(th) > ESCAPE] = np.nan  # văng ra ngoài thì dừng hẳn
    ok = np.abs(dL(th)) < 1e-6  # NaN cho False
    print(
        f"từ θ₀ = {vn(t_start, 1)}: hội tụ với mọi η ≤ {vn(etas[ok].max(), 2)} (về θ = {vn(th[ok][-1], 3)}), "
        f"không hội tụ với mọi η từ {vn(etas[~ok].min(), 2)} đến 1 (sau {vn(20_000, 0)} bước);  2/λ của đáy đó = {vn(2 / lam, 3)}"
    )
    assert ok[etas < 2 / lam].all() and not ok[etas > 2 / lam].any()  # ranh giới đúng ở 2/λ

Ranh giới đo được trùng đúng với $2/\lambda$ của đáy mà $\theta$ lăn về. Cùng một $\eta = 0{,}6$ là ổn ở đáy cục bộ
(0,6 < 0,734) nhưng quá lớn ở đáy toàn cục (0,6 > 0,551): **tốc độ học "đúng" phụ thuộc vào độ cong của nơi ta đang đứng**,
không phải một hằng số của bài toán.

### Hai chiều: cái bát dài và hẹp
Với nhiều tham số, độ cong khác nhau theo từng hướng. Bát đơn giản nhất có hai độ cong
$\lambda_1 = 1$ (thoải) và $\lambda_2 = 10$ (dốc):

$$\mathcal{L}(\theta) = \tfrac{1}{2}\left(\lambda_1\theta_1^2 + \lambda_2\theta_2^2\right), \qquad
\nabla\mathcal{L}(\theta) = (\lambda_1\theta_1,\; \lambda_2\theta_2).$$

Mỗi tọa độ tự nhân với $1 - \eta\lambda_i$ của riêng nó. Một $\eta$ chung phải đủ nhỏ cho hướng dốc nhất:
$\eta < 2/\lambda_{\max} = 0{,}2$. Nhưng khi đó hướng thoải lại đi rất chậm.

In [ ]:
LAM = np.array([1.0, 10.0])  # hai độ cong của cái bát
TH0 = np.array([-8.0, 2.0])  # điểm xuất phát


def bowl(theta):
    return 0.5 * np.sum(LAM * theta**2)


def gd_bowl(eta, steps, theta0=TH0):
    path = [theta0.copy()]
    for _ in range(steps):
        path.append(path[-1] - eta * LAM * path[-1])  # θ ← θ − η∇L, với ∇L = λ ⊙ θ
    return np.array(path)


assert np.allclose(num_grad(bowl, TH0), LAM * TH0)  # gradient viết tay khớp sai phân
g = LAM * TH0
cos = (-g) @ (-TH0) / (np.linalg.norm(g) * np.linalg.norm(TH0))
print(
    f"tại θ = (−8; 2): −∇L = ({vn(-g[0], 0)}; {vn(-g[1], 0)}), lệch {vn(np.degrees(np.arccos(cos)), 1)}° so với hướng thẳng về đáy"
)

**Gradient không chỉ vào đáy.** Nó chỉ hướng xuống dốc *nhanh nhất tại chỗ*, và ở cái bát hẹp, hướng đó lệch 54,2° so với
đường thẳng tới đáy, vì thành dốc ($\lambda_2 = 10$) kéo mạnh hơn nhiều. Chỉ khi mọi hướng cong như nhau (bát tròn),
$-\nabla\mathcal{L}$ mới chỉ thẳng vào tâm.

### 🤔 Dự đoán trước
Ngưỡng là $\eta < 0{,}2$. Muốn thu khoảng cách tới đáy $\|\theta\|$ đi một triệu lần, $\eta$ nào cần ít bước nhất: một
$\eta$ nhỏ an toàn như 0,1 (đúng $1/\lambda_2$, triệt tiêu hướng dốc trong một bước), hay $\eta$ lớn nhất có thể, sát 0,2?

In [ ]:
def steps_to(eta, tol=1e-6, cap=20_000):
    """Số bước để ‖θ‖ nhỏ đi 1/tol lần (None nếu không bao giờ)."""
    th, n0 = TH0.copy(), np.linalg.norm(TH0)
    for k in range(cap):
        if np.linalg.norm(th) < tol * n0:
            return k
        th = th - eta * LAM * th
        if np.linalg.norm(th) > 1e6 * n0:
            return None
    return None


eta_best = 2 / (LAM.min() + LAM.max())  # = 2/11
for eta in [0.01, 0.05, 0.1, eta_best, 0.19, 0.199, 0.2, 0.21]:
    k = steps_to(eta)
    print(
        f"η = {vn(eta, 3)}:  hệ số |1 − ηλ| = {vn(abs(1 - eta * LAM[0]), 3)} và {vn(abs(1 - eta * LAM[1]), 3)}  →  {vn(k, 0) + ' bước' if k else 'không bao giờ'}"
    )
kappa = LAM.max() / LAM.min()
assert steps_to(eta_best) == int(np.ceil(np.log(1e-6) / np.log((kappa - 1) / (kappa + 1)))) == 69
assert np.abs(gd_bowl(0.2, 40)[:, 1]).tolist() == [2.0] * 41  # η = 2/λ₂: θ₂ đổi dấu mãi, không nhỏ đi chút nào

Không phải cái nào cả. $\eta = 0{,}1$ xóa hướng dốc ngay bước đầu nhưng để hướng thoải chỉ co 0,9 mỗi bước: 131 bước.
Sát ngưỡng thì hướng dốc gần như không co: $\eta = 0{,}199$ cần 1.234 bước, và đúng $\eta = 0{,}2$ thì $\theta_2$ đổi dấu
mãi giữa $+2$ và $-2$. Tốt nhất là $\eta = 2/(\lambda_1 + \lambda_2) = 2/11 \approx 0{,}182$, nơi hai hệ số bằng nhau về độ
lớn ($0{,}818$): 69 bước. Hệ số đó là $(\kappa - 1)/(\kappa + 1)$ với $\kappa = \lambda_{\max}/\lambda_{\min} = 10$ (số điều
kiện): bát càng hẹp, gradient descent càng chậm, dù chọn $\eta$ khéo đến đâu.

Hình dưới vẽ đường đồng mức của cái bát (elip xám, đáy ở dấu cộng) và 25 bước đầu với bốn $\eta$. Với 0,182, đường đi
zíc-zắc qua lại thành dốc trong khi trôi chậm dọc đáy hẹp; với 0,21 nó văng ra khỏi khung.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-9, 9, 301), np.linspace(-3.5, 3.5, 201))
levels = [0.5 * k**2 for k in (0.5, 1, 2, 3, 4, 6, 8)]
fig, axes = plt.subplots(2, 2, figsize=(10, 5.6), sharex=True, sharey=True)
for ax, (eta, name) in zip(
    axes.flat, [(0.02, "quá nhỏ"), (0.1, "1/λ₂"), (eta_best, "2/11, nhanh nhất"), (0.21, "> 2/λ₂")]
):
    ax.contour(g1, g2, 0.5 * (LAM[0] * g1**2 + LAM[1] * g2**2), levels=levels, colors=GREY, linewidths=0.8)
    p = gd_bowl(eta, 25)
    p = p[np.all(np.abs(p) < [9, 3.5], axis=1)]
    ax.plot(p[:, 0], p[:, 1], "-o", color=ORANGE, ms=3.5, lw=1.2)
    ax.plot(*TH0, "s", color=BLUE, ms=8)
    ax.plot(0, 0, "+", color="black", ms=12, mew=2)
    ax.set_title(f"η = {vn(eta, 3)} ({name})", fontsize=10)
for ax in axes[1]:
    ax.set_xlabel("θ₁ (độ cong 1)")
for ax in axes[:, 0]:
    ax.set_ylabel("θ₂ (độ cong 10)")
plt.tight_layout()
plt.show()

(Ô vuông xanh là điểm xuất phát, chấm cam là các bước.) Hai trục không cùng tỉ lệ (trục dọc bị giãn ra), nên thung lũng
trông bớt hẹp hơn thật. Đồ thị tiếp theo vẽ số bước theo $\eta$ (trục dọc dạng log): đáy của đường cong ở 2/11, và đường
gạch đứng là ngưỡng 0,2.

In [ ]:
eta_grid = np.linspace(0.004, 0.2, 50)[:-1]
ks = [steps_to(e) for e in eta_grid]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.semilogy(eta_grid, ks, "-o", color=BLUE, ms=3)
ax.axvline(0.2, color=GREY, ls="--")
ax.plot(eta_best, steps_to(eta_best), "D", color=ORANGE, ms=9, label=f"η = 2/11: {steps_to(eta_best)} bước")
ax.set(xlabel="tốc độ học η", ylabel="số bước để ‖θ‖ nhỏ đi 10⁶ lần", title="Cái bát λ = (1; 10): chậm ở hai đầu")
ax.legend(frameon=False)
plt.show()

## 6 · Điểm yên ngựa và vùng phẳng

Gradient bằng 0 không chỉ ở đáy. Hàm hai chiều dưới đây có một **điểm yên ngựa** ở gốc tọa độ: theo hướng $\theta_1$ nó
là đáy, theo hướng $\theta_2$ nó là đỉnh (giống yên ngựa: lõm theo chiều dọc lưng ngựa, lồi theo chiều ngang).
Hai đáy thật nằm ở $(0, \pm 1)$, với $f = -0{,}25$.

$$f(\theta) = \theta_1^2 + \tfrac{1}{4}\theta_2^4 - \tfrac{1}{2}\theta_2^2, \qquad
\nabla f(\theta) = \left(2\theta_1,\; \theta_2^3 - \theta_2\right).$$

Tại gốc, độ cong là $+2$ theo $\theta_1$ và $-1$ theo $\theta_2$. Ta chạy gradient descent ($\eta = 0{,}1$) từ $(2, 0)$,
từ một điểm lệch khỏi đó $10^{-8}$, và với gradient có thêm nhiễu nhỏ (độ lệch chuẩn 0,01), giống gradient của một
mini-batch.

In [ ]:
def saddle(t):
    return t[0] ** 2 + t[1] ** 4 / 4 - t[1] ** 2 / 2


def saddle_grad(t):
    return np.array([2 * t[0], t[1] ** 3 - t[1]])


def gd_saddle(theta0, eta=0.1, steps=300, sigma=0.0, seed=0):
    rng = np.random.default_rng(seed)
    path = [np.array(theta0, dtype=float)]
    for _ in range(steps):
        g = saddle_grad(path[-1]) + sigma * rng.normal(size=2)  # sigma > 0: gradient có nhiễu
        path.append(path[-1] - eta * g)
    return np.array(path)


def escape_step(path, level=0.1):
    """Bước đầu tiên mà |θ₂| vượt level, tức đã rời khỏi vùng quanh điểm yên ngựa."""
    far = np.abs(path[:, 1]) > level
    return int(np.argmax(far)) if far.any() else None


assert np.allclose(num_grad(saddle, [0.7, -0.4]), saddle_grad(np.array([0.7, -0.4])))  # gradient viết tay khớp sai phân
exact = gd_saddle([2, 0])
tiny = gd_saddle([2, 1e-8])
print(f"từ (2; 0):       sau 300 bước θ = ({sci(exact[-1, 0])}; {vn(exact[-1, 1], 1)}), f = {sci(saddle(exact[-1]))}")
print(f"từ (2; 10⁻⁸):    rời điểm yên ngựa ở bước {escape_step(tiny)}, f cuối = {vn(saddle(tiny[-1]), 4)}")
pred = np.log(0.1 / 1e-8) / np.log(1 + 0.1 * 1)  # gần gốc, θ₂ nhân (1 + η·1) mỗi bước
print(f"   dự đoán: θ₂ nhân 1,1 mỗi bước, từ 10⁻⁸ lên 0,1 mất ln(10⁷)/ln(1,1) = {vn(pred, 1)} bước")
noisy = [escape_step(gd_saddle([2, 0], sigma=0.01, seed=s)) for s in range(5)]
print(f"từ (2; 0) có nhiễu σ = 0,01, 5 seed: rời điểm yên ngựa ở bước {noisy}")
assert np.all(exact[:, 1] == 0) and abs(exact[-1, 0]) < 1e-20  # đúng θ₂ = 0 thì gradient không bao giờ đẩy θ₂
assert abs(escape_step(tiny) - pred) <= 2

Xuất phát đúng trên đường $\theta_2 = 0$, thành phần thứ hai của gradient luôn bằng 0, nên gradient descent trượt thẳng
vào điểm yên ngựa và đứng yên ở đó: $f \to 0$, cao hơn hẳn đáy $-0{,}25$. Lệch đi $10^{-8}$ thì nó thoát được, nhưng phải
mất khoảng 170 bước loanh quanh trên một **vùng phẳng** gần gốc, nơi gradient gần bằng 0 và mỗi bước chỉ nhân $\theta_2$ lên
$1{,}1$ lần. Nhiễu nhỏ đẩy $\theta$ ra khỏi đường nguy hiểm ngay từ đầu, nên thoát sau 40–73 bước tùy seed.

Hình trái vẽ đường đồng mức của $f$ (điểm yên ngựa là dấu nhân, hai đáy là dấu cộng) và ba đường đi; hình phải vẽ
$f(\theta_k)$ theo bước $k$. Đoạn nằm ngang ở $f \approx 0$ là vùng phẳng: nhìn đường cong mất mát, ta dễ tưởng đã hội tụ.

In [ ]:
noisy0 = gd_saddle([2, 0], sigma=0.01, seed=0)
cases = [
    (exact, "từ (2; 0)", GREY, "^", ":"),
    (tiny, "từ (2; 10⁻⁸)", BLUE, "o", "-"),
    (noisy0, "từ (2; 0), có nhiễu", ORANGE, "s", "--"),
]
s1, s2 = np.meshgrid(np.linspace(-2.2, 2.2, 221), np.linspace(-1.6, 1.6, 161))
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.8), gridspec_kw={"width_ratios": [1.15, 1]})
a.contour(s1, s2, s1**2 + s2**4 / 4 - s2**2 / 2, levels=np.linspace(-0.24, 4, 14), colors=GREY, linewidths=0.7)
for path, label, color, marker, ls in cases:
    a.plot(path[:, 0], path[:, 1], ls=ls, color=color, lw=1.8, marker=marker, markevery=15, ms=5, label=label)
    b.plot(saddle(path.T), ls=ls, color=color, lw=2, marker=marker, markevery=15, ms=5, label=label)
a.plot(0, 0, "x", color="black", ms=10, mew=2)
a.plot([0, 0], [-1, 1], "+", color="black", ms=12, mew=2)
a.set(xlabel="θ₁", ylabel="θ₂", title="Đường đi quanh điểm yên ngựa (0; 0)")
b.axhline(-0.25, color="black", lw=0.8)
b.text(300, -0.23, "đáy: f = −0,25", ha="right", fontsize=9)
b.set(xlabel="bước k", ylabel="f(θₖ)", ylim=(-0.3, 1), title="Mất mát theo bước: vùng phẳng ở f ≈ 0")
b.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

### Trong nhiều chiều, điểm yên ngựa nhiều hơn hẳn đáy
Tại một điểm có gradient bằng 0, mỗi hướng riêng của độ cong (mỗi trị riêng của **ma trận Hessian**, ma trận các đạo hàm
bậc hai) có thể dương hoặc âm. Đáy cần *tất cả* đều dương. Sách *Deep Learning* (Chương 8) so sánh với việc tung đồng xu:
trong $n$ chiều, cần $n$ lần ngửa liên tiếp. Thử với ma trận đối xứng ngẫu nhiên, đóng vai một Hessian "bất kỳ":

In [ ]:
rng = np.random.default_rng(0)
DRAWS = 100_000
for n in range(1, 7):
    A = rng.normal(size=(DRAWS, n, n))
    H = (A + A.transpose(0, 2, 1)) / np.sqrt(2)  # ma trận đối xứng ngẫu nhiên (phân bố GOE)
    frac = np.mean((np.linalg.eigvalsh(H) > 0).all(axis=1))
    print(f"n = {n}: mọi trị riêng dương ở {pct(frac, 3):>8} số ma trận   (tung đồng xu: {pct(0.5**n, 3)})")
    if n == 2:
        assert abs(frac - (2 - np.sqrt(2)) / 4) < 0.003  # giá trị chính xác cho n = 2: (2 − √2)/4
print(f"(mỗi n {vn(DRAWS, 0)} ma trận; với n = 2 giá trị chính xác là (2 − √2)/4 = {pct((2 - np.sqrt(2)) / 4, 3)})")

Tỉ lệ còn giảm nhanh hơn tung đồng xu, vì các trị riêng của một ma trận ngẫu nhiên đẩy nhau ra: với $n = 6$ chỉ còn 0,001%
(1 trong 100.000 ma trận đã thử) có đủ 6 trị riêng dương, so với 1,562% của đồng xu. Dean & Majumdar (2006) tính được rằng xác suất này giảm như
$e^{-0{,}27\,n^2}$. Một mạng nơ-ron có hàng triệu tham số, nên một điểm có gradient bằng 0 ở vùng mất mát cao gần như chắc
chắn là điểm yên ngựa chứ không phải đáy (Dauphin et al., 2014). Đây là lập luận cho các hàm ngẫu nhiên; với mạng thật nó
vẫn là một giả thuyết được nghiên cứu, nhưng khớp với những gì người ta đo được.

Tin tốt: gradient descent xuất phát ngẫu nhiên gần như không bao giờ dừng *đúng* tại một điểm yên ngựa (Lee et al., 2016),
vì như ta vừa thấy, chỉ cần lệch một chút là thoát. Cái giá là **thời gian** trên những vùng phẳng quanh đó, và nhiễu của
SGD giúp rút ngắn nó.

## 7 · Hồi quy softmax trên MNIST: một gradient viết tay

Giờ đến một hàm mất mát thật. Mô hình đơn giản nhất cho 10 chữ số là một lớp $z = xW + b$ (Bài 01) với softmax và
cross-entropy (Bài 04): $W$ cỡ $784 \times 10$, $b$ cỡ $10$, tổng cộng 7.850 tham số $\theta = (W, b)$. Với $N$ ảnh xếp thành
các hàng của $X$:

$$\mathcal{L}(W, b) = \frac{1}{N}\sum_{i=1}^{N} -\log \hat y_{i, y_i}, \qquad \hat y_i = \operatorname{softmax}(x_i W + b).$$

Gradient của nó gọn đến bất ngờ. Với một ảnh, $\ell = \log\sum_j e^{z_j} - z_y$, nên $\partial \ell/\partial z_k = \hat y_k - y_k$
($y$ là nhãn one-hot): xác suất dự đoán trừ đáp án. Vì $z_k = \sum_i x_i W_{ik} + b_k$, mỗi $W_{ik}$ nhận phần
$x_i\,(\hat y_k - y_k)$. Lấy trung bình trên $N$ ảnh, viết bằng ma trận:

$$\nabla_W \mathcal{L} = \frac{1}{N} X^\top (\hat Y - Y), \qquad \nabla_b \mathcal{L} = \frac{1}{N}\sum_{i} (\hat y_i - y_i).$$

Đây là một bước của quy tắc chuỗi; Bài 06 làm điều đó cho mạng nhiều lớp. Hàm `ce` dưới đây trả về mất mát và
$\partial\mathcal{L}/\partial Z = (\hat Y - Y)/N$; `softmax_loss_grad` nhân thêm $X^\top$.

In [ ]:
def ce(Z, y):
    """Cross-entropy trung bình của các hàng logit Z, và ∂L/∂Z = (Ŷ − Y)/N."""
    Z = Z - Z.max(axis=1, keepdims=True)  # trừ max của từng hàng: softmax không đổi, exp không tràn (Bài 04)
    E = np.exp(Z)
    S = E.sum(axis=1, keepdims=True)
    n = len(Z)
    loss = np.mean(np.log(S[:, 0]) - Z[np.arange(n), y])
    D = E / S  # Ŷ: xác suất softmax, mỗi hàng một ảnh
    D[np.arange(n), y] -= 1  # Ŷ − Y
    return loss, D / n


def softmax_loss_grad(W, b, X, y):
    loss, D = ce(X @ W + b, y)
    return loss, X.T @ D, D.sum(axis=0)  # ∇W = Xᵀ(Ŷ − Y)/N, ∇b = tổng các cột


def full_loss(W, b):
    return ce(Xtr @ W + b, ytr)[0]  # mất mát trên cả 60.000 ảnh huấn luyện


def accuracy(W, b):
    return np.mean((Xte @ W + b).argmax(axis=1) == yte)  # tỉ lệ đúng trên 10.000 ảnh kiểm tra


W0, b0 = np.zeros((784, 10)), np.zeros(10)
print(
    f"{vn(W0.size + b0.size, 0)} tham số; lúc W = 0, b = 0: mất mát = {vn(full_loss(W0, b0), 4)} = ln 10, đúng {pct(accuracy(W0, b0))}"
)
assert W0.size + b0.size == 7850 and np.isclose(full_loss(W0, b0), np.log(10))

Lúc $W = 0$, mọi chữ số được 10% xác suất, nên mất mát đúng bằng $\ln 10$ (Bài 04). Trước khi dùng công thức gradient, ta
kiểm tra nó với sai phân trung tâm trên **cả 7.850 tham số**, tại một $W$ ngẫu nhiên và 100 ảnh (15.700 lần tính mất mát):

In [ ]:
rng = np.random.default_rng(1)
Wr, br = rng.normal(0, 0.05, (784, 10)), rng.normal(0, 0.05, 10)
Xs, ys = Xtr[:100], ytr[:100]
_, gW, gb = softmax_loss_grad(Wr, br, Xs, ys)
g_formula = np.r_[gW.ravel(), gb]
theta_r = np.r_[Wr.ravel(), br]  # 7.850 tham số trải thành một vectơ: W trước, b sau
g_numeric = num_grad(lambda th: ce(Xs @ th[:7840].reshape(784, 10) + th[7840:], ys)[0], theta_r)
rel = np.abs(g_numeric - g_formula).max() / np.abs(g_formula).max()
print(
    f"{vn(g_formula.size, 0)} đạo hàm riêng: lệch lớn nhất {sci(np.abs(g_numeric - g_formula).max())}, tương đối {sci(rel)}"
)
assert rel < 1e-6, "công thức gradient sai"
print(f"({vn(2 * g_formula.size, 0)} lần tính mất mát cho sai phân)")

Công thức khớp sai phân tới khoảng 9 chữ số. Từ đây ta tin `softmax_loss_grad` và dùng nó; còn `num_grad` thì quá chậm để
huấn luyện (mỗi gradient tốn 15.700 lần tính mất mát thay vì một lần).

## 8 · Gradient descent trên cả tập: tốc độ học nào là quá lớn?

Mục 5 nói ngưỡng là $\eta < 2/\lambda_{\max}$, với $\lambda_{\max}$ là độ cong lớn nhất. Với 7.850 tham số, độ cong theo một
hướng $V$ đo được bằng đúng mẹo sai phân của mục 3, áp dụng cho gradient:

$$\text{độ cong theo } V \approx \Big\langle \frac{\nabla\mathcal{L}(\theta + \varepsilon V) - \nabla\mathcal{L}(\theta - \varepsilon V)}{2\varepsilon},\, V \Big\rangle, \qquad \|V\| = 1.$$

Tại $W = 0$ có một lối tắt: mọi ảnh có cùng $\hat y_k = 0{,}1$, nên độ cong tách thành tích của hai phần: phần theo ảnh
là trị riêng của ma trận $M = \tilde X^\top \tilde X / N$ (với $\tilde x = [x, 1]$, vì $b$ là trọng số của một đầu vào luôn
bằng 1), phần theo 10 lớp là $0{,}1$. Vậy $\lambda_{\max} = 0{,}1 \times$ trị riêng lớn nhất của $M$. Ô dưới tính nó, rồi đo
độ cong theo đúng hướng đó bằng sai phân để kiểm tra.

In [ ]:
Xb = np.c_[Xtr, np.ones(N)]  # x̃ = [x, 1]
eigval, eigvec = np.linalg.eigh(Xb.T @ Xb / N)
lam_max = 0.1 * eigval[-1]
u = eigvec[:, -1]  # hướng "ảnh" cong nhất
c = np.array([1.0, -1.0] + [0.0] * 8) / np.sqrt(2)  # hướng "lớp": tăng z của chữ số 0, giảm z của chữ số 1
VW, Vb = np.outer(u[:784], c), u[784] * c  # một hướng có độ dài 1 trong không gian 7.850 tham số
eps = 1e-4
_, gWp, gbp = softmax_loss_grad(W0 + eps * VW, b0 + eps * Vb, Xtr, ytr)
_, gWm, gbm = softmax_loss_grad(W0 - eps * VW, b0 - eps * Vb, Xtr, ytr)
curv = (np.sum((gWp - gWm) * VW) + np.sum((gbp - gbm) * Vb)) / (2 * eps)
print(
    f"trị riêng lớn nhất của M = {vn(eigval[-1], 4)}  →  λ_max = {vn(lam_max, 4)},  ngưỡng 2/λ_max = {vn(2 / lam_max, 3)}"
)
print(f"độ cong đo bằng sai phân theo hướng đó: {vn(curv, 4)}")
assert abs(curv - lam_max) / lam_max < 1e-6

Ngưỡng ở $W = 0$ là $\eta \approx 0{,}513$. Ta chạy gradient descent trên **cả 60.000 ảnh** mỗi bước, 20 bước, với ba tốc
độ học: ngay dưới ngưỡng ($0{,}5$), gấp đôi và gấp bốn.

In [ ]:
def gd_full(eta, steps=20):
    """Gradient descent trên cả tập huấn luyện, từ W = 0. Trả về W, b và mất mát trước mỗi bước (và sau bước cuối)."""
    W, b = np.zeros((784, 10)), np.zeros(10)
    losses = []
    for _ in range(steps):
        loss, gW, gb = softmax_loss_grad(W, b, Xtr, ytr)  # gradient trên cả 60.000 ảnh
        losses.append(loss)
        W -= eta * gW
        b -= eta * gb
    losses.append(full_loss(W, b))
    return W, b, np.array(losses)


gd_runs = {eta: gd_full(eta) for eta in [0.5, 1.0, 2.0]}
for eta, (W_, b_, Ls) in gd_runs.items():
    up = int(np.sum(np.diff(Ls) > 0))
    print(
        f"η = {vn(eta, 1)}: mất mát {vn(Ls[0], 3)} → {vn(Ls[-1], 3)} sau 20 bước, cao nhất {vn(Ls.max(), 3)} (bước {Ls.argmax()}), "
        f"{up} bước đi lên; kiểm tra đúng {pct(accuracy(W_, b_))}"
    )
L05, L2 = gd_runs[0.5][2], gd_runs[2.0][2]
assert np.all(np.diff(L05) < 0) and L2.max() > np.log(10)

Hình dưới vẽ mất mát trên cả tập huấn luyện sau mỗi bước, cho ba tốc độ học (tròn, vuông, tam giác); đường gạch là
$\ln 10$, mất mát lúc xuất phát.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
for (eta, (_, _, Ls)), color, m, ls in zip(gd_runs.items(), [BLUE, GREEN, ORANGE], ["o", "s", "^"], ["-", "--", ":"]):
    ax.plot(Ls, marker=m, ls=ls, color=color, lw=1.8, ms=5, label=f"η = {vn(eta, 1)}")
ax.axhline(np.log(10), color=GREY, ls="--", lw=1)
ax.set(xlabel="bước (mỗi bước dùng cả 60.000 ảnh)", ylabel="mất mát huấn luyện", xticks=range(0, 21, 2))
ax.set_title(f"Gradient descent trên cả tập; ngưỡng ở W = 0 là 2/λ_max = {vn(2 / lam_max, 3)}", fontsize=11)
ax.legend(frameon=False)
plt.show()

Với $\eta = 0{,}5$, ngay dưới ngưỡng, mất mát giảm ở mọi bước. Gấp đôi hay gấp bốn ngưỡng thì có những bước mất mát *tăng*:
với $\eta = 2$ nó vọt lên cao gấp gần ba lần lúc xuất phát. Nhưng khác với cái bát của mục 5, nó không phân kỳ mãi mà
lại đi xuống. Một lý do: cross-entropy không phải một cái bát. Ở xa đáy nó chỉ tăng gần như tuyến tính, nên gradient của
nó bị chặn: với mọi logit, $\|\hat y - y\| \le \sqrt{2}$, nên gradient của một ảnh không dài quá $\sqrt{2}\,\|\tilde x\|$.
Mỗi bước chỉ dịch $\theta$ một đoạn có giới hạn, không bị nhân lên mãi như trong cái bát (hay như hàm bậc 4 của demo, có
gradient tăng như $\theta^3$). Độ cong cũng đổi theo vị trí, nên ngưỡng $2/\lambda$ chỉ đúng **gần một điểm**. Với mạng
nơ-ron thật, Cohen et al. (2021) thấy gradient descent trên cả tập thường tự đưa mình tới đúng mép đó: độ cong lớn nhất
tăng dần rồi lơ lửng quanh $2/\eta$ (họ gọi là *edge of stability*).

Có một vấn đề lớn hơn: 20 bước ở trên đã tính gradient trên $20 \times 60.000$ ảnh, mà mất mát vẫn còn 0,602 với
$\eta = 0{,}5$. Mục sau làm khác: mỗi bước chỉ nhìn một nhúm ảnh.

## 9 · Mini-batch SGD: bước và lượt (epoch)

**SGD theo lô nhỏ (mini-batch SGD)** thay gradient trên cả tập bằng gradient trên $B$ ảnh chọn ngẫu nhiên. Cách làm chuẩn
(cũng là cách `reel/mnist_mlp/train.py` của khóa học huấn luyện MLP): mỗi **lượt (epoch)** xáo trộn thứ tự 60.000 ảnh rồi
cắt thành các lô liền nhau, mỗi lô một **bước (iteration)**. Mỗi ảnh được dùng đúng một lần mỗi lượt.

- Gradient descent trên cả tập: 1 bước = 1 lượt.
- SGD với $B = 64$: 1 lượt = $\lceil 60.000 / 64 \rceil$ bước, lô cuối chỉ còn phần dư.

In [ ]:
def sgd(eta, B=64, epochs=1, seed=0, checkpoints=()):
    """Mini-batch SGD cho hồi quy softmax, từ W = 0. Trả về W, b và [(bước, mất mát trên cả tập)] tại các checkpoint."""
    W, b = np.zeros((784, 10)), np.zeros(10)
    rng = np.random.default_rng(seed)
    curve, step = [], 0
    for _ in range(epochs):
        order = rng.permutation(N)  # mỗi lượt xáo lại thứ tự ảnh
        for i in range(0, N, B):
            if step in checkpoints:
                curve.append((step, full_loss(W, b)))
            idx = order[i : i + B]  # một lô: B ảnh kế tiếp trong thứ tự đã xáo
            _, gW, gb = softmax_loss_grad(W, b, Xtr[idx], ytr[idx])
            W -= eta * gW
            b -= eta * gb
            step += 1
    curve.append((step, full_loss(W, b)))
    return W, b, curve


steps_per_epoch = len(range(0, N, 64))
print(f"B = 64: {steps_per_epoch} bước mỗi lượt; lô cuối có {N - 64 * (steps_per_epoch - 1)} ảnh")
assert steps_per_epoch == 938

### 🤔 Dự đoán trước
Cùng $\eta = 0{,}5$, cùng **20 bước**. Gradient descent trên cả tập dùng 60.000 ảnh mỗi bước; SGD dùng 64 ảnh mỗi bước.
Sau 20 bước, bên nào có mất mát huấn luyện thấp hơn? Và sau khi mỗi bên đã dùng *cùng một lượng ảnh*, một lượt?

In [ ]:
cps = set(range(21)) | {int(k) for k in np.geomspace(1, 937, 14)}
W_sgd, b_sgd, curve = sgd(0.5, checkpoints=cps)
sgd_loss = dict(curve)
print(f"sau 20 bước:  GD trên cả tập {vn(L05[20], 3)}  ·  SGD (B = 64) {vn(sgd_loss[20], 3)}")
print(f"sau 1 lượt:   GD trên cả tập {vn(L05[1], 3)} (1 bước)  ·  SGD (B = 64) {vn(sgd_loss[938], 3)} (938 bước)")
print(
    f"GD cần 20 lượt để còn {vn(L05[20], 3)}; SGD còn {vn(sgd_loss[938], 3)} sau 1 lượt, kiểm tra đúng {pct(accuracy(W_sgd, b_sgd))}"
)
assert L05[20] < sgd_loss[20] and sgd_loss[938] < L05[20]
assert abs(accuracy(W_sgd, b_sgd) - 0.9172) < 0.0005

Tính theo **bước**, gradient descent trên cả tập thắng một chút: 0,602 so với 0,694, vì mỗi bước của nó đi đúng hướng
dốc nhất. Nhưng mỗi bước đó tốn gấp 938 lần. Tính theo **lượng ảnh đã dùng**, thứ quyết định thời gian chạy, SGD thắng
xa: sau một lượt nó đã xuống 0,296, thấp hơn cả 20 lượt của gradient descent.

Hình dưới vẽ cả hai cách nhìn. Trái: mất mát theo bước. Phải: cùng dữ liệu đó theo số ảnh đã dùng để tính gradient (trục
log). Gradient descent là đường xanh với chấm tròn, SGD là đường cam với ô vuông.

In [ ]:
sgd_steps = np.array(sorted(sgd_loss))
sgd_vals = np.array([sgd_loss[k] for k in sgd_steps])
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
a.plot(range(21), L05, "-o", color=BLUE, ms=4, label="GD trên cả tập")
a.plot(sgd_steps[sgd_steps <= 20], sgd_vals[sgd_steps <= 20], "--s", color=ORANGE, ms=4, label="SGD, B = 64")
a.set(xlabel="bước", ylabel="mất mát huấn luyện", title="Theo bước", xticks=range(0, 21, 5))
b.semilogx(np.arange(1, 21) * N, L05[1:], "-o", color=BLUE, ms=4, label="GD trên cả tập")
b.semilogx(sgd_steps[1:] * 64, sgd_vals[1:], "--s", color=ORANGE, ms=4, label="SGD, B = 64")
b.axvline(N, color=GREY, lw=1, ls=":")
b.text(N * 1.1, 2.1, "1 lượt", fontsize=9)
b.set(xlabel="số ảnh đã dùng để tính gradient", title="Theo lượng tính toán")
a.legend(frameon=False)
plt.tight_layout()
plt.show()

Lý do SGD hiệu quả: gradient trên cả tập là một trung bình của 60.000 gradient từng ảnh, và các ảnh MNIST giống nhau rất
nhiều (hàng nghìn số 1 gần như trùng nhau). Một lô 64 ảnh đã cho một ước lượng đủ tốt của trung bình đó, với giá rẻ hơn
938 lần. Mục 10 đo xem "đủ tốt" là tốt đến đâu.

### Tốc độ học cho SGD
Quét $\eta$ với SGD, $B = 64$, một lượt, seed 0:

In [ ]:
for eta in [0.001, 0.01, 0.1, 0.5, 1.0, 100.0]:
    W_, b_, cv = sgd(eta)
    print(f"η = {vn(eta, 3):>7}: mất mát huấn luyện {vn(cv[-1][1], 3):>6}, kiểm tra đúng {pct(accuracy(W_, b_))}")
    if eta == 0.1:
        assert abs(accuracy(W_, b_) - 0.9089) < 0.0005
    if eta == 100.0:
        assert cv[-1][1] > np.log(10) and accuracy(W_, b_) > 0.7  # mất mát tệ hơn lúc xuất phát, độ chính xác vẫn khá
seed_acc = [accuracy(W_sgd, b_sgd)] + [accuracy(*sgd(0.5, seed=s)[:2]) for s in (1, 2)]  # seed 0: mô hình của mục 9
print(f"η = 0,5 với 3 seed: kiểm tra đúng {pct(min(seed_acc))} – {pct(max(seed_acc))}")

- **Quá nhỏ** ($0{,}001$): một lượt chỉ đưa mất mát từ 2,303 xuống 1,587. Đi đúng hướng nhưng như rùa bò.
- **Vùng tốt** ($0{,}5$ đến $1$): mất mát 0,296–0,298, đúng 91,44%–91,72%. Khác biệt giữa $\eta = 0{,}5$ và $1$ nằm
  trong dao động giữa các seed (91,34% – 91,72% với $\eta = 0{,}5$).
- **Quá lớn** ($\eta = 100$): mất mát cuối lượt còn cao hơn lúc xuất phát ($\ln 10$), vậy mà độ chính xác vẫn khá. Mỗi bước
  nhảy xa đến mức các logit rất lớn: softmax gần như đặt hết xác suất vào một chữ số, và mô hình **tự tin mà sai** trên
  nhiều ảnh, đúng loại lỗi cross-entropy phạt nặng nhất (Bài 04). Khi đó $\hat Y - Y$ chỉ còn là "trừ ảnh khỏi chữ số đoán
  nhầm, cộng vào chữ số đúng": quy tắc perceptron của Bài 01. Với $\eta$ lớn như vậy, kết quả còn thay đổi theo seed và theo
  cả sai số làm tròn của máy, nên con số in ra trên máy bạn có thể khác.

Nhìn độ chính xác không đủ để chọn $\eta$; phải nhìn cả mất mát.

## 10 · Gradient của một lô tốt đến đâu?

Gradient của lô $g_B$ là trung bình của $B$ gradient từng ảnh chọn ngẫu nhiên, nên trung bình của nó đúng bằng gradient
thật $g$ (ước lượng không chệch); chỉ có nhiễu. Theo luật sai số chuẩn của trung bình mẫu, độ lệch điển hình
$\|g_B - g\|$ giảm như $1/\sqrt{B}$: lô lớn gấp 100 lần chỉ bớt nhiễu 10 lần.

Với hồi quy softmax, gradient của từng ảnh là tích ngoài $\tilde x_i^\top(\hat y_i - y_i)$, nên độ dài bình phương của nó
là tích $\|\tilde x_i\|^2\,\|\hat y_i - y_i\|^2$. Nhờ vậy ta tính được *chính xác* phương sai của nhiễu trên cả 60.000 ảnh,
không cần lấy mẫu, và so với đo thực tế trên 100 lô ngẫu nhiên mỗi cỡ.

In [ ]:
SIZES_B = [1, 4, 16, 64, 256, 1024, 4096]


def batch_noise(W, b, draws=100, seed=2):
    """Độ lệch tương đối ‖g_B − g‖/‖g‖ (đo và lý thuyết) và cos(g_B, g), cho mỗi cỡ lô B."""
    _, D = ce(Xtr @ W + b, ytr)
    g = np.r_[(Xtr.T @ D).ravel(), D.sum(axis=0)]  # gradient thật, trên cả 60.000 ảnh
    sq = (np.sum(Xtr**2, axis=1) + 1) * np.sum((N * D) ** 2, axis=1)  # ‖gᵢ‖² = ‖x̃ᵢ‖² · ‖ŷᵢ − yᵢ‖²
    sigma2 = sq.mean() - g @ g  # phương sai của gradient một ảnh
    rng = np.random.default_rng(seed)
    out = []
    for B in SIZES_B:
        err, cos = [], []
        for _ in range(draws):
            idx = rng.choice(N, B, replace=False)
            _, bW, bb = softmax_loss_grad(W, b, Xtr[idx], ytr[idx])
            gB = np.r_[bW.ravel(), bb]
            err.append(np.sum((gB - g) ** 2))
            cos.append(gB @ g / (np.linalg.norm(gB) * np.linalg.norm(g)))
        theory = np.sqrt(sigma2 / B * (N - B) / (N - 1)) / np.linalg.norm(g)  # chọn không hoàn lại
        out.append((B, np.sqrt(np.mean(err)) / np.linalg.norm(g), theory, np.mean(cos)))
    return np.array(out)


noise0 = batch_noise(W0, b0)
for B, rel, th, cos in noise0:
    print(
        f"W = 0,  B = {int(B):>4}: ‖g_B − g‖/‖g‖ đo {vn(rel, 3):>6}, lý thuyết {vn(th, 3):>6};  cos(g_B, g) = {vn(cos, 2)}"
    )
assert np.all(np.abs(noise0[2:, 1] / noise0[2:, 2] - 1) < 0.1)  # B ≥ 16: đo khớp lý thuyết trong 10%

Lúc xuất phát, một lô 64 ảnh đã cho gradient lệch khoảng bằng độ dài của chính gradient thật, nhưng vẫn cùng hướng
(cos 0,69). Với 100 lô mỗi cỡ, phép đo khớp lý thuyết trong vài phần trăm ở mọi $B$.

### 🤔 Dự đoán trước
Giờ lấy mô hình sau một lượt SGD ở mục 9 (mất mát 0,296). Ở đó, gradient của một lô 64 ảnh còn gần cùng hướng với
gradient thật không? Đoán cos của chúng: gần 1, khoảng 0,7 như lúc đầu, hay nhỏ hơn nhiều?

In [ ]:
noise1 = batch_noise(W_sgd, b_sgd)
for B, rel, th, cos in noise1:
    print(
        f"sau 1 lượt, B = {int(B):>4}: ‖g_B − g‖/‖g‖ đo {vn(rel, 2):>6}, lý thuyết {vn(th, 2):>6};  cos(g_B, g) = {vn(cos, 2)}"
    )
assert np.all(np.abs(noise1[2:, 1] / noise1[2:, 2] - 1) < 0.1)
assert noise1[3, 3] < 0.3  # B = 64: gradient của lô gần như vuông góc với gradient thật

Nhỏ hơn nhiều: cos chỉ khoảng 0,15, và độ lệch gấp gần 6 lần độ dài gradient thật. (Với $B = 1$ và $4$, số đo thấp hơn lý
thuyết: sau khi học, vài ảnh khó có gradient rất lớn, và 100 lô hiếm khi bốc trúng chúng.) Gần đáy, gradient thật nhỏ đi (các
ảnh "kéo" về những hướng khác nhau và gần triệt tiêu nhau), còn gradient từng ảnh thì không, nên **nhiễu lấn át tín hiệu**.

SGD vẫn đi xuống được vì nhiễu trung bình bằng 0: qua $k$ bước, phần tín hiệu cộng dồn theo $k$ còn phần nhiễu chỉ lớn
theo $\sqrt{k}$. Nhưng với $\eta$ cố định, nó không bao giờ dừng hẳn ở đáy mà lảng vảng quanh đó. Robbins & Monro (1951) chỉ
ra rằng muốn hội tụ thì $\eta$ phải giảm dần sao cho $\sum_k \eta_k = \infty$ và $\sum_k \eta_k^2 < \infty$ (ví dụ
$\eta_k \propto 1/k$). Lịch giảm tốc độ học và các bộ tối ưu như momentum, Adam là nội dung Bài 07.

Hình dưới vẽ độ lệch tương đối theo cỡ lô (hai trục log): chấm là đo, đường là lý thuyết; tròn xanh tại $W = 0$, vuông
cam sau một lượt. Hai đường song song với độ dốc $-\tfrac{1}{2}$, đúng luật $1/\sqrt{B}$.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for res, color, m, name in [(noise0, BLUE, "o", "lúc W = 0"), (noise1, ORANGE, "s", "sau 1 lượt SGD")]:
    ax.loglog(res[:, 0], res[:, 2], color=color, lw=1.5)
    ax.loglog(res[:, 0], res[:, 1], m, color=color, ms=7, label=name)
ax.axhline(1, color=GREY, ls="--", lw=1)
ax.text(1.1, 0.8, "nhiễu = tín hiệu", ha="left", fontsize=9)
ax.set(xlabel="cỡ lô B", ylabel="‖g_B − g‖ / ‖g‖", title="Nhiễu của gradient lô giảm như 1/√B")
ax.legend(frameon=False)
plt.show()

## 11 · Cùng vòng lặp cho mạng của khóa học: 784 → 128 → 64 → 10

Khóa học dùng một ví dụ xuyên suốt: mạng MLP 784 → 128 → 64 → 10 của MLSysBook (ReLU, softmax + cross-entropy, khởi tạo
He), huấn luyện bằng chính mini-batch SGD vừa viết: $\eta = 0{,}1$, $B = 64$, seed 0, 5 lượt
(`reel/mnist_mlp/train.py`). Vòng lặp không đổi gì; chỉ có hàm tính gradient là phức tạp hơn.

Ba dòng "lượt ngược" trong ô dưới là **lan truyền ngược**, nội dung của Bài 06. Ở đây ta chưa suy ra chúng; ta chỉ làm điều
mục 3 đã dạy: không tin một công thức gradient trước khi kiểm tra nó bằng sai phân.

In [ ]:
SIZES = [784, 128, 64, 10]


def mlp_init(seed=0):
    """Khởi tạo He như train.py; trả về cả rng, vì train.py dùng tiếp nó để xáo thứ tự ảnh."""
    rng = np.random.default_rng(seed)
    params = [
        [rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out)), np.zeros(n_out)] for n_in, n_out in itertools.pairwise(SIZES)
    ]
    return params, rng


def mlp_forward(params, x):
    (W1, b1), (W2, b2), (W3, b3) = params
    h1 = np.maximum(0, x @ W1 + b1)  # z = xW + b rồi ReLU (Bài 02–03)
    h2 = np.maximum(0, h1 @ W2 + b2)
    return h1, h2, h2 @ W3 + b3


def mlp_loss_grad(params, x, y):
    h1, h2, z = mlp_forward(params, x)
    loss, dz = ce(z, y)  # dz = (Ŷ − Y)/N, đúng như mục 7
    (_, _), (W2, _), (W3, _) = params
    # lượt ngược: Bài 06 suy ra hai dòng này bằng quy tắc chuỗi; ở đây ta chỉ kiểm tra chúng bằng sai phân
    dh2 = (dz @ W3.T) * (h2 > 0)
    dh1 = (dh2 @ W2.T) * (h1 > 0)
    return loss, [(x.T @ dh1, dh1.sum(0)), (h1.T @ dh2, dh2.sum(0)), (h2.T @ dz, dz.sum(0))]


params, _ = mlp_init(seed=0)
n_params = sum(W.size + b.size for W, b in params)
print(f"{vn(n_params, 0)} tham số; gradient bằng sai phân cần {vn(2 * n_params, 0)} lần tính mất mát cho mỗi bước")
assert n_params == 109386

Kiểm tra 30 đạo hàm riêng chọn ngẫu nhiên (5 phần tử của mỗi $W$ và mỗi $b$ trong ba lớp), trên 100 ảnh:

In [ ]:
def loss_with(P, idx, value, x, y):
    """Mất mát khi đặt đúng một tham số P[idx] = value (rồi trả lại giá trị cũ)."""
    old = P[idx]
    P[idx] = value
    loss = mlp_loss_grad(params, x, y)[0]
    P[idx] = old
    return loss


xs, ys = Xtr[:100], ytr[:100]
_, grads = mlp_loss_grad(params, xs, ys)
rng = np.random.default_rng(3)
worst = 0.0
for layer, which in itertools.product(range(3), range(2)):  # which = 0: W, 1: b
    P = params[layer][which]
    for _ in range(5):
        idx = tuple(int(rng.integers(0, n)) for n in P.shape)
        g_num = num_grad(lambda v, P=P, idx=idx: loss_with(P, idx, v[0], xs, ys), [P[idx]])[0]
        worst = max(worst, abs(g_num - grads[layer][which][idx]))
print(f"30 đạo hàm riêng: lệch lớn nhất giữa công thức và sai phân {sci(worst)}")
assert worst < 1e-7

Lượt ngược khớp. Giờ chạy đúng một lượt của `train.py`: cùng khởi tạo, cùng thứ tự ảnh, ghi mất mát trung bình của mỗi 50
lô như tệp dữ liệu của khóa học (`web/src/content/mnist-mlp.json`, khóa `history`) rồi so với các giá trị trong đó.

In [ ]:
JSON_HISTORY = [1.2031, 0.5253, 0.4288, 0.3964, 0.3448, 0.328, 0.3182, 0.2595, 0.2644]  # mnist-mlp.json, bước 50–450
JSON_HISTORY += [0.262, 0.2222, 0.2438, 0.2219, 0.2178, 0.1976, 0.2091, 0.2098, 0.2072]  # bước 500–900
params, rng = mlp_init(seed=0)
order = rng.permutation(N)  # lượt 1
batch_losses = []
t0 = time.perf_counter()
for i in range(0, N, 64):
    idx = order[i : i + 64]
    loss, grads = mlp_loss_grad(params, Xtr[idx], ytr[idx])
    batch_losses.append(loss)
    for layer, (gW, gb) in zip(params, grads):
        layer[0] -= 0.1 * gW  # θ ← θ − η∇L, cho từng ma trận W
        layer[1] -= 0.1 * gb  # và từng vectơ b
hist = np.array(batch_losses[: 50 * (len(batch_losses) // 50)]).reshape(-1, 50).mean(axis=1)
acc_mlp = np.mean(mlp_forward(params, Xte)[2].argmax(axis=1) == yte)
print(f"{len(batch_losses)} bước trong {vn(time.perf_counter() - t0, 1)} giây; kiểm tra đúng {pct(acc_mlp)} sau 1 lượt")
print("mất mát mỗi 50 bước:", " · ".join(vn(v, 4) for v in hist[:6]), "…", vn(hist[-1], 4))
print(f"lệch lớn nhất so với mnist-mlp.json: {vn(np.abs(np.round(hist, 4) - JSON_HISTORY).max(), 4)}")
assert np.abs(hist - JSON_HISTORY).max() < 5e-4 and abs(acc_mlp - 0.9427) < 0.0005, "khác với dữ liệu của khóa học"

Đúng từng con số của khóa học: sau một lượt, mạng đúng 94,27% ảnh kiểm tra (`train.py` chạy tiếp 4 lượt nữa để đạt
97,27%, khóa `testAcc` của `mnist-mlp.json`, con số mà các demo từ Bài 04 dùng). Cùng một lượt, hồi quy softmax của mục 9 chỉ đạt 91,72%: nó chỉ có một lớp
tuyến tính.

Hình dưới vẽ mất mát của **từng lô** (đường xám mảnh, 938 bước) và trung bình mỗi 50 bước (chấm xanh). Mỗi lô chỉ có 64
ảnh, nên mất mát của nó nhảy lên xuống rất mạnh quanh xu hướng đi xuống: đó là nhiễu của SGD mà mục 10 đã đo.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(np.arange(1, len(batch_losses) + 1), batch_losses, color=GREY, lw=0.6, label="mất mát của từng lô (64 ảnh)")
ax.plot(np.arange(50, 50 * len(hist) + 1, 50), hist, "o", color=BLUE, ms=6, label="trung bình mỗi 50 bước")
ax.set(xlabel="bước (1 lượt = 938 bước)", ylabel="cross-entropy", ylim=(0, 2.6), xlim=(0, 940))
ax.set_title("Mạng 784 → 128 → 64 → 10, một lượt SGD (η = 0,1; B = 64; seed 0)", fontsize=11)
ax.legend(frameon=False)
plt.show()

## 12 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### Điểm xuất phát quyết định thung lũng
Với hàm của demo và $\eta = 0{,}15$, đổi $\theta_0$:

In [ ]:
for start in [-3.0, -1.0, 0.1, 0.18, 0.2, 0.5, 3.0]:
    tr, st = gd_1d(0.15, start, max_steps=500)
    where = "toàn cục" if tr[-1] < 0 else "cục bộ"
    print(f"θ₀ = {vn(start, 2):>5}: {st} sau {len(tr) - 1:>3} bước ở θ = {vn(tr[-1], 3):>6} (đáy {where})")

Ranh giới giữa hai thung lũng nằm ở đỉnh $\theta \approx 0{,}189$: xuất phát bên trái đỉnh thì về đáy toàn cục, bên phải
thì về đáy cục bộ. Xuất phát càng gần đỉnh, độ dốc càng gần 0, nên càng mất nhiều bước.

### Cỡ lô, với cùng $\eta = 0{,}5$ và một lượt

In [ ]:
for B in [8, 16, 64, 256, 1024]:
    W_, b_, cv = sgd(0.5, B=B)
    print(
        f"B = {B:>4}: {vn(cv[-1][0], 0):>5} bước mỗi lượt, mất mát {vn(cv[-1][1], 3)}, kiểm tra đúng {pct(accuracy(W_, b_))}"
    )

Lô lớn cho gradient ít nhiễu hơn, nhưng một lượt lại có ít bước hơn, nên với cùng lượng tính toán nó đi được ít hơn.
Lô quá nhỏ với $\eta = 0{,}5$ thì nhiễu quá lớn: mỗi bước dịch tham số một đoạn cỡ $\eta$ nhân độ lệch của gradient lô, mà
độ lệch đó lớn như $1/\sqrt{B}$. Trong khoảng này, $B = 64$ là tốt nhất (với seed 0).

### Lô lớn hơn, bước dài hơn
Goyal et al. (2017) huấn luyện ImageNet với lô 8.192 ảnh bằng **quy tắc tỉ lệ tuyến tính**: nhân $B$ lên $k$ lần thì nhân
$\eta$ lên $k$ lần, để một lượt vẫn đi được quãng đường tương tự. Thử từ $B = 64$, $\eta = 0{,}1$:

In [ ]:
for k in [1, 4, 16, 64]:
    W_, b_, cv = sgd(0.1 * k, B=64 * k)
    print(
        f"B = {64 * k:>4}, η = {vn(0.1 * k, 1):>3}: {cv[-1][0]:>3} bước, mất mát {vn(cv[-1][1], 3)}, kiểm tra đúng {pct(accuracy(W_, b_))}"
    )

Quy tắc giữ được kết quả tới $B = 256$ và gần như tới $1.024$, rồi gãy ở $B = 4.096$: khi đó $\eta = 6{,}4$ lớn hơn ngưỡng
$2/\lambda_{\max} \approx 0{,}51$ của mục 8 hơn mười lần, và một lượt chỉ còn 15 bước để kịp hồi lại. Goyal et al. cũng phải
thêm *warmup* (tăng $\eta$ dần trong vài lượt đầu) mới dùng được lô lớn.

## 13 · Giới hạn: gradient của mạng nhiều lớp từ đâu ra?

Gradient descent chỉ cần một thứ: $\nabla\mathcal{L}(\theta)$. Với hồi quy softmax, ta viết được nó bằng tay trong một dòng
($X^\top(\hat Y - Y)/N$) vì chỉ có một lớp. Với mạng 784 → 128 → 64 → 10, ta đã dùng hai dòng "lượt ngược" mà chưa giải
thích, và chỉ kiểm tra được chúng bằng sai phân: muốn tính cả gradient bằng sai phân phải tính mất mát 218.772 lần cho
*mỗi bước*, so với một lượt xuôi và một lượt ngược. Bài 06 suy ra những dòng đó từ quy tắc chuỗi.

Còn một giới hạn nữa: với $\eta$ cố định, SGD không bao giờ dừng hẳn ở đáy vì nhiễu (mục 10), và một $\eta$ chung cho mọi
hướng thì chậm trong thung lũng hẹp (mục 5). Momentum, Adam và lịch giảm tốc độ học là nội dung Bài 07.

## Tóm tắt
- Gradient là vectơ các đạo hàm riêng, đo được bằng sai phân trung tâm; dùng nó để kiểm tra mọi công thức (mục 3, 7, 11).
- Một bước: $\theta \leftarrow \theta - \eta\,\nabla\mathcal{L}(\theta)$. Trên hàm của demo, từ $\theta_0 = 2{,}6$: $\eta = 0{,}02$
  cần 126 bước, $0{,}15$ cần 13, $0{,}9$ nhảy qua lại mãi, $1{,}3$ văng ra sau 3 bước.
- Gần một đáy có độ cong $\lambda$, gradient descent ổn định khi $\eta < 2/\lambda$ (0,551 và 0,734 ở hai đáy của demo; 0,513
  cho hồi quy softmax tại $W = 0$). Thung lũng hẹp ($\kappa = 10$) buộc phải đi zíc-zắc: 69 bước ở $\eta$ tốt nhất.
- Gradient chỉ hướng dốc nhất tại chỗ, không chỉ vào đáy (lệch 54,2° trong cái bát hẹp), và bằng 0 cả ở điểm yên ngựa.
  Trong nhiều chiều, điểm yên ngựa nhiều hơn hẳn đáy.
- Mini-batch SGD: một lượt $B = 64$ là 938 bước rẻ, đưa mất mát xuống 0,296; 20 lượt gradient descent trên cả tập mới
  xuống 0,602.
- Nhiễu của gradient lô giảm như $1/\sqrt{B}$, và gần đáy thì lớn hơn chính gradient (cos ≈ 0,15 với $B = 64$).
- Cùng vòng lặp huấn luyện mạng 784 → 128 → 64 → 10 của khóa học: 94,27% sau một lượt, đúng như dữ liệu của khóa học.

## Câu hỏi ôn tập
1. Vì sao ta bước theo $-\nabla\mathcal{L}$ chứ không phải $+\nabla\mathcal{L}$?
2. Gần một đáy có độ cong $\lambda = 4$, tốc độ học lớn nhất còn ổn định là bao nhiêu? Với $\eta = 0{,}4$ thì $\theta$ đi tới
   đáy thế nào?
3. Vì sao trong cái bát $\lambda = (1; 10)$, $\eta$ tốt nhất không phải $1/\lambda_2 = 0{,}1$?
4. Gradient descent xuất phát đúng tại $(2; 0)$ của hàm yên ngựa thì dừng ở đâu? Vì sao một chút nhiễu lại giúp ích?
5. Một lượt của SGD với $B = 64$ trên 60.000 ảnh là bao nhiêu bước? Gradient descent trên cả tập thì sao?
6. Lô lớn gấp 100 lần thì nhiễu của gradient lô nhỏ đi bao nhiêu lần?

<details><summary>Gợi ý đáp án</summary>

1. $\nabla\mathcal{L}$ chỉ hướng mất mát tăng nhanh nhất; ngược chiều nó là hướng giảm nhanh nhất (với bước đủ nhỏ).
2. $2/\lambda = 0{,}5$. Với $\eta = 0{,}4$, hệ số $1 - \eta\lambda = -0{,}6$: khoảng cách tới đáy co còn 60% mỗi bước nhưng
   đổi dấu, nên $\theta$ nhảy qua lại hai bên đáy trong khi tiến lại gần.
3. Với $\eta = 0{,}1$, hướng thoải chỉ co $0{,}9$ mỗi bước. $\eta = 2/11$ cân bằng hai hướng (cùng hệ số $0{,}818$ về độ lớn),
   nên hướng chậm nhất nhanh hơn.
4. Tại điểm yên ngựa $(0; 0)$: thành phần $\theta_2$ của gradient luôn bằng 0 trên đường $\theta_2 = 0$, nên không gì đẩy nó ra.
   Nhiễu đưa $\theta_2$ khác 0, và độ cong âm theo hướng đó khuếch đại nó cho tới khi thoát.
5. $\lceil 60.000/64 \rceil = 938$ bước. Gradient descent trên cả tập: 1 bước mỗi lượt.
6. Khoảng 10 lần: nhiễu tỉ lệ với $1/\sqrt{B}$.
</details>

## Tìm hiểu thêm
- C. Lemaréchal (2012), [Cauchy and the Gradient Method](https://doi.org/10.4171/dms/6/27): Cauchy đề xuất phương pháp năm 1847.
- H. Robbins & S. Monro (1951), [A Stochastic Approximation Method](https://doi.org/10.1214/aoms/1177729586): gốc của SGD.
- Goodfellow, Bengio & Courville, [*Deep Learning*, Chương 8](https://www.deeplearningbook.org/contents/optimization.html):
  mini-batch, điểm yên ngựa, vùng phẳng.
- Y. Dauphin et al. (2014), [Identifying and attacking the saddle point problem](https://arxiv.org/abs/1406.2572).
- D. Dean & S. Majumdar (2006), [Large deviations of extreme eigenvalues of random matrices](https://doi.org/10.1103/PhysRevLett.97.160201):
  xác suất mọi trị riêng đều dương.
- J. Lee et al. (2016), [Gradient Descent Converges to Minimizers](https://arxiv.org/abs/1602.04915): khởi tạo ngẫu nhiên gần như
  chắc chắn tránh được điểm yên ngựa; S. Du et al. (2017), [Gradient Descent Can Take Exponential Time to Escape Saddle Points](https://arxiv.org/abs/1705.10412).
- J. Cohen et al. (2021), [Gradient Descent on Neural Networks Typically Occurs at the Edge of Stability](https://arxiv.org/abs/2103.00065).
- P. Goyal et al. (2017), [Accurate, Large Minibatch SGD](https://arxiv.org/abs/1706.02677): quy tắc tỉ lệ tuyến tính và warmup.
- [Dive into Deep Learning, mục Gradient Descent](https://d2l.ai/chapter_optimization/gd.html) và
  [CS231n, Optimization](https://cs231n.github.io/optimization-1/).

**Bài tiếp theo:** [Bài 06 · Lan truyền ngược](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/lan-truyen-nguoc/): suy ra gradient
của mạng nhiều lớp bằng quy tắc chuỗi, với giá cỡ một lượt xuôi.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")